# Работа с пропусками в pandas

Короткий разбор на данных этого же проекта: как найти пропуски, когда их заполнять,
когда удалять строки, и чем `dropna` отличается от `drop_duplicates`.
Отдельно — почему построчный `apply` в аналитике заменяют векторной операцией.

In [1]:
import pandas as pd
import numpy as np

user_level = pd.read_csv('../data/user_level.csv')
daily = pd.read_csv('../data/daily.csv')

print(f"user_level: {user_level.shape}")
print(f"daily:      {daily.shape}")

user_level: (100000, 8)
daily:      (118, 6)


## С чего начинаю всегда: гранулярность

Прежде чем считать что-либо на пользователя, надо убедиться, что строка — это и есть
пользователь. Если уникальных идентификаторов меньше, чем строк, значит строка это событие
или пользователь-день, и любую метрику на человека сначала надо свернуть.

In [2]:
print(f"Уникальных user_id: {user_level['user_id'].nunique()}")
print(f"Строк в таблице:    {len(user_level)}")
print(f"Строка = пользователь: {user_level['user_id'].nunique() == len(user_level)}")

Уникальных user_id: 100000
Строк в таблице:    100000
Строка = пользователь: True


## Есть ли пропуски

`isna()` даёт таблицу True/False того же размера, `.sum()` схлопывает её по колонкам.
В этих файлах пропусков нет — данные уже прошли обработку.

In [3]:
print("Пропуски по колонкам user_level:")
print(user_level.isna().sum())
print(f"\nВсего пропусков: user_level {user_level.isna().sum().sum()}, daily {daily.isna().sum().sum()}")

Пропуски по колонкам user_level:
user_id               0
group                 0
total_revenue         0
total_sessions        0
total_time            0
total_transactions    0
active_days           0
is_payer              0
dtype: int64

Всего пропусков: user_level 0, daily 0


## Откуда пропуски берутся на практике

Чаще всего они не приходят из файла, а появляются после джойна: у строк левой таблицы
не нашлось пары в правой. Воспроизведу это — приклею к полной таблице срез только по
группе A, и у строк группы B появятся пустые значения.

In [4]:
daily_a = (daily[daily['group'] == 'A']
           [['date', 'group', 'revenue']]
           .rename(columns={'revenue': 'revenue_a'}))

daily_gaps = daily.merge(daily_a, on=['date', 'group'], how='left')

print(f"Строк после джойна: {len(daily_gaps)}")
print(f"Пропусков в revenue_a: {daily_gaps['revenue_a'].isna().sum()}")

Строк после джойна: 118
Пропусков в revenue_a: 59


## fillna — когда ноль осмыслен

Здесь пропуск означает «строки не было в правой таблице», то есть ноль выручки по
определению. Такой пропуск заполнять нулём корректно.

А вот пропуск в дате, в идентификаторе или в категории означал бы «мы не знаем».
Заполнять его нельзя: это выдумывание данных, и оно молча исказит все дальнейшие расчёты.

In [5]:
daily_filled = daily_gaps.copy()
daily_filled['revenue_a'] = daily_filled['revenue_a'].fillna(0)

print(f"Пропусков было:  {daily_gaps['revenue_a'].isna().sum()}")
print(f"Пропусков стало: {daily_filled['revenue_a'].isna().sum()}")

Пропусков было:  59
Пропусков стало: 0


## dropna и drop_duplicates — разные задачи

Их легко перепутать на слух, но они решают противоположные проблемы:

* `dropna` убирает строки, где **нет значения** — борется с неполнотой данных;
* `drop_duplicates` убирает строки, которые **повторяют** другие — борется с задвоением.

Ниже обе операции рядом, чтобы разница была видна на числах.

In [6]:
# dropna: выбрасываем строки без значения в указанной колонке
daily_no_na = daily_gaps.dropna(subset=['revenue_a'])

print(f"dropna(subset=['revenue_a']): {len(daily_no_na)} строк из {len(daily_gaps)}")
print(f"Выброшено {len(daily_gaps) - len(daily_no_na)} строк — это ровно группа B")
print(f"Остались группы: {daily_no_na['group'].unique().tolist()}")

dropna(subset=['revenue_a']): 59 строк из 118
Выброшено 59 строк — это ровно группа B
Остались группы: ['A']


In [7]:
# drop_duplicates: другая задача — задвоение строк
print(f"Полных дубликатов:        {user_level.duplicated().sum()}")
print(f"Дубликатов по user_id:    {user_level.duplicated(subset=['user_id']).sum()}")
print(f"drop_duplicates() оставит: {len(user_level.drop_duplicates())} из {len(user_level)}")

Полных дубликатов:        0
Дубликатов по user_id:    0
drop_duplicates() оставит: 100000 из 100000


## apply против векторной операции

`apply(axis=1)` вызывает функцию Python для каждой строки. Работает, но на ста тысячах
строк это заметно. Та же задача — выручка на сессию — решается делением колонки на колонку.

In [8]:
def revenue_per_session(row):
    if row['total_sessions'] > 0:
        return row['total_revenue'] / row['total_sessions']
    return 0.0

%timeit user_level.apply(revenue_per_session, axis=1)

616 ms ± 15.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
safe_sessions = user_level['total_sessions'].replace(0, np.nan)

%timeit np.where(user_level['total_sessions'] > 0, user_level['total_revenue'] / safe_sessions, 0.0)

450 μs ± 31.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [10]:
# Убеждаемся, что оба способа дают одно и то же
by_apply = user_level.apply(revenue_per_session, axis=1)
by_vector = np.where(user_level['total_sessions'] > 0,
                     user_level['total_revenue'] / safe_sessions, 0.0)

print(f"Результаты совпадают: {np.allclose(by_apply.values, by_vector)}")

Результаты совпадают: True


**Вывод.** Пропуски чаще всего появляются на джойне, а не в исходном файле. Заполнять их
нулём можно только там, где ноль имеет смысл; в остальных случаях строку либо удаляют,
либо оставляют пропуск как есть и учитывают его в расчёте. `apply` стоит держать для
случаев, где векторного аналога нет — разница по времени видна в замерах выше.